## Evaluate integration models - using donor as batch for hvg

In [ ]:
# Run this in terminal:
# In this specific notebook we use 'py_analysis'
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_scpoli \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_spoli \
#        --display-name "scpoli"'

## Load Libraries

In [3]:
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc
import subprocess
from joblib import parallel_backend
from matplotlib.backends.backend_pdf import PdfPages
import hdf5plugin

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [4]:
# Directories
ma.create_directories(dir_path = str(here('data/integration/third_pass')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/tmp')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/embedding')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/files')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/plot')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/objects')))
ma.create_directories(dir_path = str(here('data/integration/third_pass/models')))

/work/islet_cartography_scrna/data/integration/third_pass Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/tmp Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/embedding Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/files Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/plot Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/objects Directory already exists!
/work/islet_cartography_scrna/data/integration/third_pass/models Directory already exists!


In [5]:
# Saving
base_dir = str(here('data/integration/third_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
emb_dir = os.path.join(base_dir, 'embedding') 
objects_dir = os.path.join(base_dir, 'objects') 

In [6]:
hvgs = [500, 1000, 2000]
methods = ["scpoli"]
adata_dir = tmp_dir  # directory with preprocessed HVG files
file_dir = os.path.join(base_dir, 'files')  # directory to save embeddings
model_dir = os.path.join(base_dir, 'models')   # directory to save models
key_batch = ["technical_integration", "ic_id_donor_integrate"]   # adjust to your obs keys
latent_dims_list = [15, 20]
embedding_dims_list = [10, 20, 30]
label_key = 'study_cell_annotation_harmonized' 

In [7]:
# plotting
sc.set_figure_params(figsize=(5, 5), frameon=False)
# save path
sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [8]:
# PAth to enviroments
py_analysis = '/work/islet_cartography_scrna/scrna_cartography_py_analysis'
scpoli = '/work/islet_cartography_scrna/scrna_cartography_scpoli'

## Load data

In [9]:
adata =sc.read_h5ad(here('data/anndata/AB_combined.h5ad'))

In [10]:
# Add technical variation
adata.obs['technical_integration'] = (
    adata.obs['cell_nuclei'].astype(str) + "_" + adata.obs['count_quantification'].astype(str)
)

# copy of raw counts
adata.raw = adata.copy()

# make na unknown, otherwise it will not work
adata.obs[label_key] = adata.obs[label_key].fillna('unknown')

In [ ]:
for n in hvgs:
    
    ad = adata.copy()

    print(f'Finding {n} highly variable genes with batch awareness: {key_batch[1]}')
    
    sc.pp.highly_variable_genes(
        ad,
        n_top_genes=n,
        flavor="seurat_v3",
        layer="counts",
        span=0.5, # as suggested in https://github.com/scverse/scanpy/issues/2669#issuecomment-1768365664 (fraction of cells used when estimating the variance in the loess model fit if)
        batch_key =  key_batch[1],
        subset=True)
    
    # Extract hvgs
    hvg_list = ad.var[ad.var['highly_variable']].index.tolist()
    
    # Save hvgs 
    hvg_path = os.path.join(file_dir, f'hvgs_{n}.txt')
    
    with open(hvg_path, 'w') as f:
        for gene in hvg_list:
            f.write(gene + '\n')
    
    print(f'Saved highly variable genes to {hvg_path}')
    
    # Save adata object
    adata_file = os.path.join(objects_dir, f"adata_{n}_hvg.h5ad")
    ad.write(adata_file)

    print(f'Saved adata object to {adata_file}')
    
    # Save adata object for scpoli
    adata_file_scpoli = os.path.join(objects_dir, f"adata_scpoli_{n}_hvg.h5ad")
    del ad.uns["log1p"]
    ad.write(adata_file_scpoli)
    
    print(f'Saved adata object for scpoli to {adata_file_scpoli}')
    
    for latent_dims in latent_dims_list:
        
        key_save = f"{'_'.join(key_batch)}_latent{latent_dims}"

        print(f'Running {key_save}')
        print(f'Running no integration')
        # Run unintegrated (PCA)
        result_int = subprocess.run([
            "conda", "run", "-p", py_analysis, "python", "run_no_int.py",
            str(n), adata_file, model_dir, emb_dir, key_save, str(latent_dims)
        ], capture_output=True, text=True)
        print(result_int.stderr)

        for embedding_dims in embedding_dims_list:
            key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"

            print(f'Running {key_save}')
         
            print(f'Running ScPoli')
            # Run scPoli
            result_scpoli = subprocess.run([
                "conda", "run", "-p", scpoli, "python", "run_scpoli.py",
                str(n), adata_file_scpoli, model_dir, emb_dir, ",".join(key_batch), key_save, str(latent_dims), str(embedding_dims)
            ], capture_output=True, text=True)
            print(result_scpoli.stderr)
            

## Add latent embedding

In [11]:
# Make dictionary of adata objects
adata_dict = {}

# Make copies per n
for n in hvgs:
    adata_dict[n] = adata.copy()

In [13]:
for n in hvgs:
    # Unintegrated embeddings
    for latent_dims in latent_dims_list:
        key_save = f"{'_'.join(key_batch)}_latent{latent_dims}"
        df_path = os.path.join(emb_dir, f"unintegrated_latent_embd_{n}_{key_save}.csv")
        obsm_key = f"unintegrated_{n}_{key_save}"
        adata_dict[n] = ma.add_embedding(adata_dict[n], df_path, obsm_key)

    # Method-specific embeddings
    for method in methods:
        for latent_dims in latent_dims_list:
            for embedding_dims in embedding_dims_list:
                key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"
                df_path = os.path.join(emb_dir, f"{method}_latent_embd_{n}_{key_save}.csv")
                obsm_key = f"{method}_{n}_{key_save}"
                adata_dict[n] = ma.add_embedding(adata_dict[n], df_path, obsm_key)

Added embedding 'unintegrated_500_technical_integration_ic_id_donor_integrate_latent15' with shape (589202, 15).
Added embedding 'unintegrated_500_technical_integration_ic_id_donor_integrate_latent20' with shape (589202, 20).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent15_embed10' with shape (589202, 15).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent15_embed20' with shape (589202, 15).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent15_embed30' with shape (589202, 15).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent20_embed10' with shape (589202, 20).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent20_embed20' with shape (589202, 20).
Added embedding 'scpoli_500_technical_integration_ic_id_donor_integrate_latent20_embed30' with shape (589202, 20).
Added embedding 'unintegrated_1000_technical_integration_ic_id_donor_integrate_laten

In [14]:
# save adata and replace with object with directory
for n in hvgs:
    dir = os.path.join(objects_dir, f'adata_{n}_integration.h5ad')
    adata_dict[n].write(
        dir,
        compression=hdf5plugin.FILTERS["zstd"],
        compression_opts=hdf5plugin.Zstd(clevel=5).filter_options)
    adata_dict[n] = dir

In [ ]:
# Get paths if you have reloaded this script
adata_dict = {}
for n in hvgs:
    dir = os.path.join(objects_dir, f'adata_{n}_integration.h5ad')
    adata_dict[n] = dir
adata_dict

In [ ]:
adata_dict

## Find neighbors and umap

In [ ]:
for n in hvgs:
    print(f"Running neighbor/UMAP integration for {n}...")

    ad = ma.neighbor_umap_integration(adata_dict[n], verbose=True, overwrite=False)

    # Save updated AnnData with new embeddings
    ad.write(
        adata_dict[n],
        compression=hdf5plugin.FILTERS["zstd"],
        compression_opts=hdf5plugin.Zstd(clevel=5).filter_options
    )

    print(f"Finished neighbors/UMAP for {n}, saved to {adata_dict[n]}\n")

## Plot UMAP

In [ ]:
# Plot UMAPs for all embeddings and save
for n in hvgs:
    ad = sc.read_h5ad(adata_dict[n])
    figures = ma.plot_umaps_for_adata(
        ad,
        colors=("technical_integration", "study_cell_annotation_harmonized", "name", "library_prep", "disease", "sex_predicted"),
        verbose=False
    )

    pdf_path = os.path.join(plot_dir, f'UMAP_{n}_integration_additional_variables.pdf')

    with PdfPages(pdf_path) as pdf:
        for umap_key, fig in figures.items():
            pdf.savefig(fig)
            plt.close(fig)  # free memory
    
    print(f"Saved {len(figures)} UMAPs to {pdf_path}")